In [ ]:
'''
    ১. Encoding কী?
    ----------------
    কম্পিউটার সরাসরি "hi" বা "hello" বোঝে না। কম্পিউটার মূলত binary বা bytes
    (যেমন: 01101000) নিয়ে কাজ করে।

    Encoding হলো এমন একটি নির্দিষ্ট নিয়ম, যার মাধ্যমে bytes-কে মানুষের পড়ার
    যোগ্য character বা text-এ রূপান্তর করা হয়।

        উদাহরণ:
        Bytes → Encoding Rule → Text


    ২. UTF-8
    --------
    UTF-8 বর্তমানে সবচেয়ে বেশি ব্যবহৃত standard text encoding।

    Python 3-তে:
    - str হলো Unicode text।
    - Source code এবং text file নিয়ে কাজ করার ক্ষেত্রে UTF-8 খুবই গুরুত্বপূর্ণ।
    - তবে "Python 3 সব string-কে UTF-8 হিসেবে রাখে" বলা পুরোপুরি সঠিক নয়;
    Python-এর str internally Unicode representation ব্যবহার করে।


    ৩. Mojibake / Character Encoding Mismatch
    ------------------------------------------
    যদি কোনো file একটি encoding-এ লেখা হয়, কিন্তু সেটিকে অন্য encoding দিয়ে
    read করার চেষ্টা করা হয়, তাহলে text-এর জায়গায় অর্থহীন বা অদ্ভুত character
    দেখাতে পারে।

    এটিকে বলা হয়:
    - Mojibake
    - Character Encoding Mismatch

    উদাহরণ:
        Windows-1252 দিয়ে লেখা data
                    ↓
            UTF-8 দিয়ে read
                    ↓
        অদ্ভুত/ভুল character


    ৪. Python-এ String এবং Bytes
    ----------------------------
    Python 3-তে text নিয়ে কাজ করার সময় মূলত দুটি data type গুরুত্বপূর্ণ:

        ১. str
        মানুষের পড়ার যোগ্য Unicode text।

        ২. bytes
        raw binary data। Python-এ bytes literal-এর আগে b থাকে।

        উদাহরণ:
        b"hello"


    ৫. encode() এবং decode()
    ------------------------
    str এবং bytes-এর মধ্যে conversion করার জন্য encode() এবং decode() ব্যবহার
    করা হয়।

        String → Bytes
        ----------------
        encode()

        Bytes → String
        ----------------
        decode()


        উদাহরণ:
        text = "hello"

        data = text.encode("utf-8")
        text_again = data.decode("utf-8")


    ৬. ভুল Encoding ব্যবহার করলে কী হয়?
    -----------------------------------
    যদি bytes-কে ভুল encoding দিয়ে decode করার চেষ্টা করা হয়, তাহলে
    UnicodeDecodeError হতে পারে।

    বিশেষ করে ASCII-এর সীমাবদ্ধতা মনে রাখতে হবে।

    ASCII মূলত English character-এর জন্য তৈরি এবং € বা Bangla-এর মতো
    অনেক character support করে না।

    তাই এমন data:

        b"€"

    কে ASCII দিয়ে decode করতে গেলে UnicodeDecodeError হতে পারে।


    ৭. charset_normalizer দিয়ে Encoding Detect করা
    -----------------------------------------------
    অনেক সময় আমরা জানি না একটি dataset বা file কোন encoding ব্যবহার করেছে।

    পুরো file পরীক্ষা করা সময়সাপেক্ষ হতে পারে। তাই charset_normalizer ব্যবহার
    করে file-এর encoding সম্পর্কে একটি ধারণা পাওয়া যায়।

    উদাহরণ:

        from charset_normalizer import detect

        with open("data.csv", "rb") as file:
            raw_data = file.read(10000)

        result = detect(raw_data)

    এখানে প্রথম 10,000 bytes analyse করে সম্ভাব্য encoding detect করা হয়।

    গুরুত্বপূর্ণ:
    - এটি encoding "নিশ্চিতভাবে" বলে না।
    - এটি সম্ভাব্য encoding সম্পর্কে একটি guess দেয়।


    ৮. Pandas-এ Encoding Error
    --------------------------
    Dataset load করার সময় এমন error দেখা যেতে পারে:

        UnicodeDecodeError

    যেমন:

        df = pd.read_csv("data.csv")

    যদি file-এর encoding UTF-8 না হয়, তাহলে Pandas default encoding দিয়ে
    read করতে গিয়ে error হতে পারে।


    ৯. Encoding Error-এর Solution
    -----------------------------
    প্রথমে file-এর encoding detect করতে হবে।

    ধরা যাক, detect করার পর পাওয়া গেল:

        Windows-1252

    তাহলে Pandas-কে explicitly encoding বলে দিতে হবে:

        df = pd.read_csv("data.csv", encoding="windows-1252")

    অর্থাৎ:

        File-এর Encoding
                ↓
        Detect / Identify
                ↓
        Pandas-কে সেই Encoding দেওয়া
                ↓
        Dataset Successfully Load


    ১০. UTF-8-এ File Save করা
    -------------------------
    একবার সঠিক encoding ব্যবহার করে dataset Pandas-এ load করার পর,
    dataset-টিকে একটি standard encoding-এ save করে রাখা ভালো।

    উদাহরণ:

        df.to_csv("new_file.csv", index=False, encoding="utf-8")

    এতে পরবর্তীতে একই dataset নিয়ে কাজ করার সময় encoding-related সমস্যা
    কমে যায়।

    Note:
    Pandas-এর current CSV writing default encoding সাধারণত UTF-8 হলেও,
    code-এ encoding="utf-8" explicitly লেখা বেশি clear এবং predictable।


    ১১. Practical ML Implication
    ----------------------------
    Machine Learning project-এ প্রথমেই dataset successfully load করতে হয়।

    যদি encoding সমস্যার কারণে dataset load-ই না হয়, তাহলে:
        Data Loading
            ↓
        Data Cleaning
            ↓
        EDA
            ↓
        Feature Engineering
            ↓
        Model Training

    কোনোটিই properly করা সম্ভব হবে না।

    তাই real-world ML data pipeline-এ বিভিন্ন encoding সম্পর্কে ধারণা থাকা
    গুরুত্বপূর্ণ। বিশেষ করে পুরোনো CSV, exported database বা বিভিন্ন system
    থেকে পাওয়া text data নিয়ে কাজ করার সময় encoding mismatch দেখা যেতে পারে।
    
'''

In [ ]:
import pandas as pd
import charset_normalizer

# ==========================================
# ধাপ ১: সমস্যার সিমুলেশন (Dataset Creation)
# ==========================================
# ধরি, আমরা একটি ডেটাসেট পেয়েছি যেখানে একটি শহরের নাম জাপানিজ ক্যারেক্টারে লেখা
data = {
    'user_id': [101, 102],
    'name': ['Jihad', 'Taro'],
    'city': ['Dhaka', '東京']  # '東京' মানে টোকিও
}
df_mock = pd.DataFrame(data)

# ফাইলটি আমরা ইচ্ছা করে 'shift_jis' (জাপানিজ এনকোডিং) দিয়ে সেভ করছি।
# রিয়েল লাইফে ক্লায়েন্ট বা সার্ভার থেকে ফাইলটি এভাবেই আসবে।
df_mock.to_csv("japan_data_raw.csv", index=False, encoding="shift_jis")


# ==========================================
# ধাপ ২: সমস্যা (The Error)
# ==========================================
# তুমি যদি সাধারণ নিয়মে এটি রিড করার চেষ্টা করো, Python একে UTF-8 ভাববে এবং ক্র্যাশ করবে।
try:
    df_error = pd.read_csv("japan_data_raw.csv")
except UnicodeDecodeError as e:
    print(f"Error Encountered: {e}\n")
    # Output: 'utf-8' codec can't decode byte 0x93...


# ==========================================
# ধাপ ৩: সমাধান (Detect & Read)
# ==========================================
# ফাইলটির আসল এনকোডিং কী, তা আমরা charset_normalizer দিয়ে বের করবো
with open("japan_data_raw.csv", 'rb') as rawdata:
    # ফাইলের প্রথম ১০০০০ বাইট রিড করে এনকোডিং গেস করা হচ্ছে
    result = charset_normalizer.detect(rawdata.read(10000))

print(f"Detected Encoding: {result['encoding']}") 
# Output: Detected Encoding: shift_jis (বা কাছাকাছি কিছু)

# এবার সঠিক এনকোডিং প্যারামিটারটি দিয়ে ফাইল রিড করবো
df_corrected = pd.read_csv("japan_data_raw.csv", encoding=result['encoding'])

print("\nSuccessfully Read Data:")
print(df_corrected)


# ==========================================
# ধাপ ৪: স্ট্যান্ডার্ডাইজেশন (Save as UTF-8)
# ==========================================
# ডেটা একবার ঠিকভাবে রিড করার পর, আমরা এটিকে গ্লোবাল স্ট্যান্ডার্ড UTF-8 এ সেভ করে রাখবো।
# Pandas ডিফল্টভাবেই utf-8 এ সেভ করে।
df_corrected.to_csv("japan_data_cleaned.csv", index=False)